In [1]:
import sys
sys.path.append("C:/Users/Drew/Documents/MyFiles/work/manim_files")
from utils import dag
import networkx as nx
import numpy as np

G=dag()
pos = nx.multipartite_layout(G, subset_key='layer')

In [2]:
pos

{0: array([-0.82142857, -0.25      ]),
 1: array([-0.82142857,  0.25      ]),
 2: array([-0.32142857, -0.75      ]),
 3: array([-0.32142857, -0.25      ]),
 4: array([-0.32142857,  0.25      ]),
 5: array([-0.32142857,  0.75      ]),
 6: array([ 0.17857143, -1.        ]),
 7: array([ 0.17857143, -0.5       ]),
 8: array([0.17857143, 0.        ]),
 9: array([0.17857143, 0.5       ]),
 10: array([0.17857143, 1.        ]),
 11: array([ 0.67857143, -0.5       ]),
 12: array([0.67857143, 0.        ]),
 13: array([0.67857143, 0.5       ])}

In [67]:
class FordFulkerson:
    def __init__(self, G:nx.DiGraph, sources:list, sinks:list):
        self.G = G
        self.sources = sources
        self.sinks = sinks
        self._initialize_supersources()

    def _initialize_supersources(self):
        self.source = 's'
        self.sink = 't'
        self.capacity = {(u,v): 1 for (u,v) in self.G.edges()}
        self.flow = {(u,v): 0 for (u,v) in self.G.edges()}
        self.steps = [{"path": None, "flow": self.flow.copy(), "delta": 0}]
        self.H = self.G.copy()
        M = len(self.G.edges())
        for s in self.sources:
            print(self.source, s)
            self.H.add_edge(self.source, s)
            self.capacity.update({(self.source, s):M})
            self.flow.update({(self.source, s):0})
        for t in self.sinks:
            self.H.add_edge(t, self.sink)
            self.capacity.update({(t, self.sink):M})
            self.flow.update({(t, self.sink):0})
            print(t, self.sink)

    def residual_graph(self):
        R = nx.DiGraph()
        R.add_nodes_from(self.H.nodes())
        for u, v in self.H.edges():
            # Forward residual capacity
            forward = self.capacity[(u, v)] - self.flow[(u, v)]
            if forward > 0:
                R.add_edge(u, v, capacity=forward)
            # Reverse residual capacity
            backward = self.flow[(u, v)]
            if backward > 0:
                R.add_edge(v, u, capacity=backward)
        return R

    def find_augmenting_path(self):
        R = self.residual_graph()
        if not nx.has_path(R, self.source, self.sink):
            return None
        return nx.shortest_path(R, self.source, self.sink)

    def augment(self, path):
        R = self.residual_graph()
        # Find bottleneck residual capacity
        delta = min(R[u][v]["capacity"] for u, v in zip(path[:-1], path[1:]))
        # Push flow along the augmenting path
        for u, v in zip(path[:-1], path[1:]):
            # Forward edge
            if (u, v) in self.capacity:
                self.flow[(u, v)] += delta
            # Reverse residual edge
            else:
                self.flow[(v, u)] -= delta
        # Save resulting flow state
        self.steps.append({"path": path, "flow": self.flow.copy(), "delta": delta})
        return delta

    def run(self):
        while True:
            path = self.find_augmenting_path()
            if path is None:
                break
            self.augment(path)
        return self.steps

    def max_flow_val(self):
        return sum(self.flow[self.source, s] for s in self.sources)

In [ ]:
arr = np.array(list(nx.get_node_attributes(G, 'layer').values()))
sinks = np.where(arr == max(arr))[0].tolist()
sources = np.where(arr == min(arr))[0].tolist()
ff = FordFulkerson(G, sources, sinks)

s 0
s 1
11 t
12 t
13 t


In [72]:
ff.steps[1]['path']

['s', 0, 5, 8, 13, 't']

In [69]:
ff.run()
for i, step in enumerate(ff.steps):
    print(f"\nStep {i}")
    print(f"Path:  {step['path']}")
    print(f"Delta: {step['delta']}")
    print(f"Flow:  {step['flow']}")


Step 0
Path:  None
Delta: 0
Flow:  {(0, 5): 0, (0, 4): 0, (0, 3): 0, (1, 2): 0, (2, 10): 0, (3, 9): 0, (4, 10): 0, (5, 8): 0, (5, 6): 0, (5, 7): 0, (6, 12): 0, (7, 13): 0, (8, 13): 0, (9, 12): 0, (10, 11): 0}

Step 1
Path:  ['s', 0, 5, 8, 13, 't']
Delta: 1
Flow:  {(0, 5): 1, (0, 4): 0, (0, 3): 0, (1, 2): 0, (2, 10): 0, (3, 9): 0, (4, 10): 0, (5, 8): 1, (5, 6): 0, (5, 7): 0, (6, 12): 0, (7, 13): 0, (8, 13): 1, (9, 12): 0, (10, 11): 0, ('s', 0): 1, ('s', 1): 0, (11, 't'): 0, (12, 't'): 0, (13, 't'): 1}

Step 2
Path:  ['s', 0, 4, 10, 11, 't']
Delta: 1
Flow:  {(0, 5): 1, (0, 4): 1, (0, 3): 0, (1, 2): 0, (2, 10): 0, (3, 9): 0, (4, 10): 1, (5, 8): 1, (5, 6): 0, (5, 7): 0, (6, 12): 0, (7, 13): 0, (8, 13): 1, (9, 12): 0, (10, 11): 1, ('s', 0): 2, ('s', 1): 0, (11, 't'): 1, (12, 't'): 0, (13, 't'): 1}

Step 3
Path:  ['s', 0, 3, 9, 12, 't']
Delta: 1
Flow:  {(0, 5): 1, (0, 4): 1, (0, 3): 1, (1, 2): 0, (2, 10): 0, (3, 9): 1, (4, 10): 1, (5, 8): 1, (5, 6): 0, (5, 7): 0, (6, 12): 0, (7, 13): 0, (8,

np.int64(3)

{0: array([-0.82142857, -0.25      ,  0.        ]),
 1: array([-0.82142857,  0.25      ,  0.        ]),
 2: array([-0.32142857, -0.75      ,  0.        ]),
 3: array([-0.32142857, -0.25      ,  0.        ]),
 4: array([-0.32142857,  0.25      ,  0.        ]),
 5: array([-0.32142857,  0.75      ,  0.        ]),
 6: array([ 0.17857143, -1.        ,  0.        ]),
 7: array([ 0.17857143, -0.5       ,  0.        ]),
 8: array([0.17857143, 0.        , 0.        ]),
 9: array([0.17857143, 0.5       , 0.        ]),
 10: array([0.17857143, 1.        , 0.        ]),
 11: array([ 0.67857143, -0.5       ,  0.        ]),
 12: array([0.67857143, 0.        , 0.        ]),
 13: array([0.67857143, 0.5       , 0.        ])}

In [8]:
import random 
sys.path.append("C:/Users/Drew/Documents/MyFiles/work")
from networks.dep_src.graph_utils import create_layered_graph, assign_properties

layer_sizes=[1,3,5,4]
G, _ = create_layered_graph(layer_sizes)
G = assign_properties(G, layer_sizes)
pos = nx.multipartite_layout(G, subset_key='layer')
for n, d in G.nodes(data=True):
    d['pos'] = pos[n]

def remove_edges(G, remove_fraction=0.6):
    layers = [data['layer'] for _, data in G.nodes(data=True) if 'layer' in data]
    if not layers:
        return G.copy()
        
    min_layer, max_layer = min(layers), max(layers)
    
    sources = {node for node, data in G.nodes(data=True) if data.get('layer') == min_layer}
    sinks = {node for node, data in G.nodes(data=True) if data.get('layer') == max_layer}
    intermediate_nodes = {node for node, data in G.nodes(data=True) if data.get('layer') not in (min_layer, max_layer)}

    def _is_valid(g):
        # 1. Ensure every source can reach at least one sink
        for s in sources:
            if not any(nx.has_path(g, s, t) for t in sinks):
                return False
                
        # 2. Ensure every sink is reachable from at least one source
        for t in sinks:
            if not any(nx.has_path(g, s, t) for s in sources):
                return False

        # 3. Ensure intermediate nodes are not orphaned
        for node in intermediate_nodes:
            if g.in_degree(node) == 0 or g.out_degree(node) == 0:
                return False
                
        return True

    H = G.copy()
    edges = list(H.edges())
    random.shuffle(edges)
    
    target_removals = int(len(edges) * remove_fraction)
    removed_count = 0

    for u, v in edges:
        if removed_count >= target_removals:
            break
        
        H.remove_edge(u, v)
        if not _is_valid(H):
            H.add_edge(u, v)  # Revert
        else:
            removed_count += 1
            
    return H

H = remove_edges(G)

